| Merchant | 4 | `avg_interaction_value`, `category_id` (sequential category index), `value_std_norm`, `user_repeat_rate` | Non-graph-redundant signals |

In [1]:
# ── Configuration ──────────────────────────────────────────────────────────────
OUT_DIR      = '../Datasets/yelp-merchant/'
RAW_DIR      = '../datasetRaw/yelp/'
DATASET_NAME = 'yelp-merchant'

import os, json, pickle
import numpy as np
import pandas as pd
from collections import defaultdict
from datetime import datetime

with open(OUT_DIR + 'user2id.pkl', 'rb') as f:
    user2id = pickle.load(f)
with open(OUT_DIR + 'merchant2id.pkl', 'rb') as f:
    merchant2id = pickle.load(f)

usrnum      = len(user2id)
merchantnum = len(merchant2id)
print(f'Loaded mappings: {usrnum:,} users, {merchantnum:,} merchants')

Loaded mappings: 26,668 users, 19,307 merchants


In [2]:
bid_to_cat: dict = {}
business_path = RAW_DIR + 'yelp_academic_dataset_business.json'

with open(business_path, 'r', encoding='utf-8') as f:
    for line in f:
        b   = json.loads(line.strip())
        bid = b.get('business_id', '')
        if bid not in merchant2id:
            continue
        cats       = b.get('categories') or ''
        cat_tokens = [c.strip() for c in cats.split(',') if c.strip()]
        chosen_cat = cat_tokens[0] if cat_tokens else 'Unknown'
        bid_to_cat[bid] = chosen_cat

unique_cats = sorted(set(bid_to_cat.values()))
cat2idx     = {cat: i + 1 for i, cat in enumerate(unique_cats) if cat != 'Unknown'}
print(f'Unique categories mapped: {len(unique_cats):,}  |  categories: {unique_cats[:20]}')

merchant_cat = np.zeros(merchantnum, dtype=np.float32)
for bid, cat in bid_to_cat.items():
    mid = merchant2id.get(bid)
    if mid is not None:
        merchant_cat[mid] = float(cat2idx.get(cat, 0))

Unique categories mapped: 600  |  categories: ['Acai Bowls', 'Accessories', 'Active Life', 'Acupuncture', 'Adult Entertainment', 'Afghan', 'African', 'Airlines', 'Airport Lounges', 'Airport Terminals', 'Airports', 'Alternative Medicine', 'American (New)', 'American (Traditional)', 'Amusement Parks', 'Animal Shelters', 'Antiques', 'Appliances', 'Appliances & Repair', 'Aquarium Services']


In [3]:
# ── Cell 3: Stream reviews → build per-user/merchant/edge statistics ───────────
def parse_ts(s: str):
    try:
        return datetime.strptime(s, '%Y-%m-%d %H:%M:%S').timestamp()
    except Exception:
        return None

u_count     = np.zeros(usrnum,      dtype=np.int64)
u_val_sum   = np.zeros(usrnum,      dtype=np.float64)
u_val_sq    = np.zeros(usrnum,      dtype=np.float64)
u_merchants = [set() for _ in range(usrnum)]
u_min_ts    = np.full(usrnum,  np.inf)
u_max_ts    = np.full(usrnum, -np.inf)

m_count     = np.zeros(merchantnum, dtype=np.int64)
m_val_sum   = np.zeros(merchantnum, dtype=np.float64)
m_val_sq    = np.zeros(merchantnum, dtype=np.float64)
m_users     = [set() for _ in range(merchantnum)]

edge_val_sum: dict = defaultdict(float)
edge_cnt:     dict = defaultdict(int)
global_max_ts      = -np.inf

n_kept = 0
review_path = RAW_DIR + 'yelp_academic_dataset_review.json'
with open(review_path, 'r', encoding='utf-8') as f:
    for line in f:
        rv      = json.loads(line.strip())
        uid_str = rv.get('user_id', '')
        bid     = rv.get('business_id', '')
        stars   = rv.get('stars')
        if uid_str not in user2id or bid not in merchant2id or stars is None:
            continue
        uid  = user2id[uid_str]
        mid  = merchant2id[bid]
        norm = float(stars) / 5.0

        u_count[uid]   += 1
        u_val_sum[uid]  += norm
        u_val_sq[uid]   += norm * norm
        u_merchants[uid].add(mid)
        m_count[mid]   += 1
        m_val_sum[mid]  += norm
        m_val_sq[mid]   += norm * norm
        m_users[mid].add(uid)

        ts = parse_ts(rv.get('date', ''))
        if ts is not None:
            if ts < u_min_ts[uid]: u_min_ts[uid] = ts
            if ts > u_max_ts[uid]: u_max_ts[uid] = ts
            if ts > global_max_ts: global_max_ts = ts

        edge_val_sum[(uid, mid)] += norm
        edge_cnt[(uid, mid)]     += 1
        n_kept += 1

print(f'Processed {n_kept:,} reviews')

Processed 677,004 reviews


In [4]:
# ── Cell 4: Build 4+4 feature arrays ──────────────────────────────────────────
# Kept  (user)    : avg_interaction_value, activity_span_days, recency_score, value_std_norm
# Kept  (merchant): avg_interaction_value, category_id (MCC), value_std_norm, user_repeat_rate
# Dropped: interaction_count, unique_*_count, txns_per_week, repeat_visit_rate, popularity_rank
# Rationale: dropped features are graph-redundant (= node degree, neighbourhood size, or
# algebraic derivatives thereof). GNN propagation already encodes them implicitly.
# Kept features provide orthogonal signal: interaction quality (avg_value), temporal
# profile (span / recency), preference volatility (value_std), and the only exogenous
# signal available (MCC merchant category).

SECS_PER_DAY  = 86400.0
SECS_PER_YEAR = 365.0 * SECS_PER_DAY

u_avg_val       = np.where(u_count > 0, u_val_sum / u_count, 0.0).astype(np.float32)
u_var           = np.maximum(u_val_sq / np.maximum(u_count, 1) - (u_val_sum / np.maximum(u_count, 1))**2, 0.0)
u_val_std       = np.sqrt(u_var).astype(np.float32)
u_unique_merch  = np.array([len(s) for s in u_merchants], dtype=np.float32)
u_span_days     = np.where(np.isfinite(u_min_ts) & np.isfinite(u_max_ts),
                            (u_max_ts - u_min_ts) / SECS_PER_DAY, 0.0).astype(np.float32)
u_recency       = np.where(np.isfinite(u_max_ts),
                            np.exp(-np.maximum(0.0, global_max_ts - u_max_ts) / SECS_PER_YEAR),
                            0.0).astype(np.float32)

m_avg_val       = np.where(m_count > 0, m_val_sum / m_count, 0.0).astype(np.float32)
m_var           = np.maximum(m_val_sq / np.maximum(m_count, 1) - (m_val_sum / np.maximum(m_count, 1))**2, 0.0)
m_val_std       = np.sqrt(m_var).astype(np.float32)
m_unique_users  = np.array([len(s) for s in m_users], dtype=np.float32)

m_repeat_u = np.zeros(merchantnum, dtype=np.float32)
for (uid, mid), cnt in edge_cnt.items():
    if cnt > 1:
        m_repeat_u[mid] += 1.0
m_user_repeat_rate = m_repeat_u / np.maximum(m_unique_users, 1.0)

user_features = np.stack([
    u_avg_val,    # 0: avg_interaction_value  (quality signal)
    u_span_days,  # 1: activity_span_days     (temporal profile)
    u_recency,    # 2: recency_score          (temporal recency)
    u_val_std,    # 3: value_std_norm         (preference volatility)
], axis=1)

merchant_features = np.stack([
    m_avg_val,          # 0: avg_interaction_value  (quality signal)
    merchant_cat,       # 1: category_id (MCC index) (only exogenous feature)
    m_val_std,          # 2: value_std_norm          (price variance)
    m_user_repeat_rate, # 3: user_repeat_rate        (loyalty signal)
], axis=1)

print(f'User features     : {user_features.shape}')
print(f'Merchant features : {merchant_features.shape}')
print(f'User   mean/col   : {user_features.mean(axis=0).round(4)}')
print(f'Merch  mean/col   : {merchant_features.mean(axis=0).round(4)}')

User features     : (26668, 4)
Merchant features : (19307, 4)
User   mean/col   : [8.1450e-01 1.2546e+03 6.8350e-01 1.9190e-01]
Merch  mean/col   : [8.018000e-01 2.980388e+02 2.030000e-01 5.600000e-02]


In [5]:
# ── Cell 5: Normalize 4+4 feature arrays ──────────────────────────────────────
# User  skip: col 0 (avg_value in [0.2,1]), col 2 (recency in [0,1])
# User  minmax-only: col 3 (value_std)
# User  log_minmax: col 1 (activity_span_days)
# Merch skip: col 0 (avg_value in [0.2,1]), col 3 (user_repeat_rate in [0,1])
# Merch minmax-only: col 2 (value_std)
# Merch log_minmax: col 1 (category_id — ordinal int index)

SKIP_NORM_USER  = {0, 2}  # avg_interaction_value, recency_score — already in [0,1]
SKIP_NORM_MERCH = {0, 3}  # avg_interaction_value, user_repeat_rate — already in [0,1]

def log_minmax(arr: np.ndarray) -> np.ndarray:
    v = np.log1p(np.clip(arr, 0, None))
    mn, mx = v.min(), v.max()
    return ((v - mn) / max(mx - mn, 1e-8)).astype(np.float32)

def minmax(arr: np.ndarray) -> np.ndarray:
    mn, mx = arr.min(), arr.max()
    return ((arr - mn) / max(mx - mn, 1e-8)).astype(np.float32)

user_features_norm  = user_features.copy()
merch_features_norm = merchant_features.copy()

for col in range(4):
    if col not in SKIP_NORM_USER:
        user_features_norm[:, col] = (
            minmax(user_features[:, col]) if col == 3       # value_std: minmax
            else log_minmax(user_features[:, col])          # activity_span: log_minmax
        )
for col in range(4):
    if col not in SKIP_NORM_MERCH:
        merch_features_norm[:, col] = (
            minmax(merchant_features[:, col]) if col == 2  # value_std: minmax
            else log_minmax(merchant_features[:, col])     # category_id: log_minmax
        )

print('After normalization:')
print(f'  User   min/max per col: {user_features_norm.min(axis=0).round(3)} / {user_features_norm.max(axis=0).round(3)}')
print(f'  Merch  min/max per col: {merch_features_norm.min(axis=0).round(3)} / {merch_features_norm.max(axis=0).round(3)}')

After normalization:
  User   min/max per col: [0.2   0.    0.135 0.   ] / [1. 1. 1. 1.]
  Merch  min/max per col: [0.2 0.  0.  0. ] / [1.  1.  1.  0.6]


In [6]:
# ── Cell 6: Edge weights — Bayesian-shrinkage on raw star ratings ──────────────
# stars/5 averages map 1-5 stars to the narrow band [0.20, 1.00] and a per-edge
# mean to [0.667, 0.857] — too compressed for GNN edge weighting.
# Bayesian shrinkage w* = (n·r̄ + c·μ) / (n+c), c=10:
#   • Deflates low-confidence (single-review) edges toward the global mean.
#   • Produces a wider spread, then min-max normalises to [0, 1].
# edge_val_sum stored stars/5 → multiply back by 5 to recover raw star means.

_pairs      = list(edge_cnt.keys())
_cnt_arr    = np.array([edge_cnt[p]                               for p in _pairs], dtype=np.float64)
_mean_stars = np.array([edge_val_sum[p] / edge_cnt[p] * 5.0      for p in _pairs], dtype=np.float64)

_global_mu  = float(_mean_stars.mean())
_C          = 10.0
_shrunk     = (_cnt_arr * _mean_stars + _C * _global_mu) / (_cnt_arr + _C)

_w_min, _w_max = float(_shrunk.min()), float(_shrunk.max())
_w_norm     = (_shrunk - _w_min) / max(_w_max - _w_min, 1e-8)

edge_weights = {p: float(_w_norm[i]) for i, p in enumerate(_pairs)}

vals = _w_norm
print(f'Edge weights: {len(edge_weights):,} pairs')
print(f'  Bayesian shrinkage: global_mu={_global_mu:.4f}, c={_C}')
print(f'  Shrunk range: [{float(_shrunk.min()):.4f}, {float(_shrunk.max()):.4f}]')
print(f'  Normalized:   min={vals.min():.4f}  max={vals.max():.4f}  mean={vals.mean():.4f}  std={vals.std():.4f}')

Edge weights: 632,313 pairs
  Bayesian shrinkage: global_mu=4.0640, c=10.0
  Shrunk range: [2.5496, 4.7473]
  Normalized:   min=0.0000  max=1.0000  mean=0.6888  std=0.0499


In [7]:
# ── Cell 7: Bipartite graph visualization ──────────────────────────────────────
# Geometric bipartite layout — users (left, blue circles) ↔ merchants (right, orange squares).
# Nodes are the top-N highest-degree users and their connected merchants.
# Edge thickness ∝ interaction weight (normalized to [0,1]).

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from collections import defaultdict as _dd

    np.random.seed(42)
    N_USR_SHOW = 20
    N_MRC_SHOW = 25

    # Index edge_weights by user
    u_to_m: dict = _dd(list)
    for (uid, mid), w in edge_weights.items():
        u_to_m[uid].append((mid, w))

    # Select highest-degree users
    top_users  = sorted(u_to_m.keys(), key=lambda u: len(u_to_m[u]), reverse=True)
    sample_u   = top_users[:N_USR_SHOW]

    # Gather connected merchants, capped at N_MRC_SHOW
    sample_m_set: set = set()
    for uid in sample_u:
        for mid, _ in u_to_m[uid]:
            sample_m_set.add(mid)
    sample_m    = sorted(sample_m_set)[:N_MRC_SHOW]
    m_visible   = set(sample_m)

    n_u, n_m = len(sample_u), len(sample_m)

    # Bipartite layout: users x=0, merchants x=1
    u_pos = {uid: (0.0, i / max(n_u - 1, 1)) for i, uid in enumerate(sample_u)}
    m_pos = {mid: (1.0, i / max(n_m - 1, 1)) for i, mid in enumerate(sample_m)}

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.axis('off')
    ax.set_title(
        f'Bipartite Interaction Graph — {DATASET_NAME}\n'
        f'(top-{n_u} users by degree  ·  {n_m} connected merchants)',
        fontsize=13, fontweight='bold', pad=14,
    )

    visible_ws = [w for uid in sample_u for mid, w in u_to_m[uid] if mid in m_visible]
    w_max = max(visible_ws) if visible_ws else 1.0

    # Draw edges
    for uid in sample_u:
        for mid, w in u_to_m[uid]:
            if mid not in m_visible:
                continue
            p1, p2 = u_pos[uid], m_pos[mid]
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]],
                    color='#90A4AE', alpha=0.4,
                    linewidth=0.4 + 2.2 * (w / w_max), zorder=1)

    # User nodes (blue circles)
    ax.scatter([u_pos[u][0] for u in sample_u], [u_pos[u][1] for u in sample_u],
               s=220, c='#1565C0', zorder=4, edgecolors='white', linewidths=1.8, label='User')
    for uid, (x, y) in u_pos.items():
        ax.text(x - 0.05, y, f'u{uid}', ha='right', va='center', fontsize=7, color='#1565C0')

    # Merchant nodes (orange squares)
    ax.scatter([m_pos[m][0] for m in sample_m], [m_pos[m][1] for m in sample_m],
               s=220, c='#E65100', marker='s', zorder=4, edgecolors='white', linewidths=1.8, label='Merchant')
    for mid, (x, y) in m_pos.items():
        ax.text(x + 0.05, y, f'm{mid}', ha='left', va='center', fontsize=7, color='#E65100')

    ax.text(0.0, -0.08, f'Users  (n={n_u})',
            ha='center', fontsize=11, color='#1565C0', fontweight='bold',
            transform=ax.transData)
    ax.text(1.0, -0.08, f'Merchants  (n={n_m})',
            ha='center', fontsize=11, color='#E65100', fontweight='bold',
            transform=ax.transData)

    ax.set_xlim(-0.22, 1.22)
    ax.set_ylim(-0.12, 1.08)
    ax.legend(loc='upper center', ncol=2, fontsize=9, framealpha=0.8)

    plt.tight_layout()
    plt.savefig(OUT_DIR + 'bipartite_graph_sample.png', bbox_inches='tight', dpi=120)
    plt.show()
    print(f'Saved: bipartite_graph_sample.png  ({n_u} users, {n_m} merchants, {len(visible_ws)} edges shown)')

except Exception as e:
    print(f'Graph visualization skipped: {e}')

Saved: bipartite_graph_sample.png  (20 users, 25 merchants, 45 edges shown)


C:\Users\User\AppData\Local\Temp\ipykernel_30192\2872773007.py:85: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# ── Cell 8: Save all files + summary ───────────────────────────────────────────
np.save(OUT_DIR + 'user_features.npy',     user_features_norm)
np.save(OUT_DIR + 'merchant_features.npy', merch_features_norm)

with open(OUT_DIR + 'edge_weights.pkl', 'wb') as f:
    pickle.dump(edge_weights, f)

meta = {
    'user_feature_names': [
        'avg_interaction_value',   # col 0
        'activity_span_days',      # col 1
        'recency_score',           # col 2
        'value_std_norm',          # col 3
    ],
    'merchant_feature_names': [
        'avg_interaction_value',   # col 0
        'category_id',             # col 1
        'value_std_norm',          # col 2
        'user_repeat_rate',        # col 3
    ],
    'edge_feature_names':  ['bayesian_shrinkage_stars'],
    'user_value_col':      0,   # avg_interaction_value is col 0 in 4+4 schema
    'merchant_value_col':  0,   # avg_interaction_value is col 0 in 4+4 schema
    'edge_reweight_strategy':    'bayesian_shrinkage',
    'edge_reweight_pseudo_count': _C,
    'edge_reweight_global_mean':  _global_mu,
    'edge_weight_range':         [float(_w_min), float(_w_max)],
    'dataset':             DATASET_NAME,
    'schema':              '4+4',
    'category_count':      len(unique_cats),
    'category_encoding':   'Yelp-category -> contiguous int index',
    'feature_selection':   'dropped graph-redundant: interaction_count, unique_merchant_count, txns_per_week, repeat_visit_rate / popularity_rank',
}
with open(OUT_DIR + 'feature_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved:')
for fname in ['user_features.npy', 'merchant_features.npy', 'edge_weights.pkl', 'feature_meta.json']:
    path = OUT_DIR + fname
    size = os.path.getsize(path) if os.path.isfile(path) else None
    status = f'{size/1024:.0f} KB' if size else 'MISSING'
    print(f'  {fname:<35} {status}')
print()
print('=' * 60)
print(f'Feature Extraction Complete — {DATASET_NAME}  [4+4 schema]')
print('=' * 60)
print(f'User feature shape     : {user_features_norm.shape}')
print(f'Merchant feature shape : {merch_features_norm.shape}')
print(f'Edge weights           : {len(edge_weights):,} pairs')
print(f'Schema (4+4)           : user_dim={user_features_norm.shape[1]}, merch_dim={merch_features_norm.shape[1]}')

Saved:
  user_features.npy                   417 KB
  merchant_features.npy               302 KB
  edge_weights.pkl                    10486 KB
  feature_meta.json                   1 KB

Feature Extraction Complete — yelp-merchant  [4+4 schema]
User feature shape     : (26668, 4)
Merchant feature shape : (19307, 4)
Edge weights           : 632,313 pairs
Schema (4+4)           : user_dim=4, merch_dim=4
